# Production Prediction Pipeline

## Objective

This notebook converts the trained customer churn model into a reusable production-oriented inference pipeline.

The objective is to establish a consistent workflow that can accept new customer data and generate:

- Churn probability
- Churn prediction
- Customer risk level
- Retention priority
- Recommended business action

The pipeline uses the production model, preprocessing pipeline, and optimized decision threshold saved during model development.

The resulting inference workflow will serve as the prediction layer for the future deployment application.

In [1]:
# Import Libraries

import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Project Paths

This is important for reproducibility.

In [2]:
# Project root
PROJECT_ROOT = Path("..")

# Model artifacts
MODEL_DIR = PROJECT_ROOT / "models"

# Output directories
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PREDICTION_DIR = OUTPUT_DIR / "predictions"

# Create directories if they do not exist
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

# Artifact paths
MODEL_PATH = MODEL_DIR / "final_churn_model.pkl"
PREPROCESSOR_PATH = MODEL_DIR / "preprocessor.pkl"
THRESHOLD_PATH = MODEL_DIR / "final_threshold.pkl"

### Load Production Artifacts
project already has the important production artifacts:

final XGBoost model
preprocessing pipeline
optimized threshold

In [3]:
# Load production model
final_model = joblib.load(MODEL_PATH)

# Load preprocessing pipeline
preprocessor = joblib.load(PREPROCESSOR_PATH)

# Load optimized classification threshold
final_threshold = joblib.load(THRESHOLD_PATH)

print("Production artifacts loaded successfully.")
print(f"Model: {MODEL_PATH}")
print(f"Preprocessor: {PREPROCESSOR_PATH}")
print(f"Threshold: {final_threshold}")

Production artifacts loaded successfully.
Model: ..\models\final_churn_model.pkl
Preprocessor: ..\models\preprocessor.pkl
Threshold: 0.33999999999999986


In [4]:
# Validate Loaded Artifacts
# Before allowing predictions, verify that the artifacts actually exist.

artifact_paths = {
    "Model": MODEL_PATH,
    "Preprocessor": PREPROCESSOR_PATH,
    "Threshold": THRESHOLD_PATH
}

for artifact_name, artifact_path in artifact_paths.items():
    print(
        f"{artifact_name}: "
        f"{'✓ Found' if artifact_path.exists() else '✗ Missing'}"
    )

# Then

if not all(path.exists() for path in artifact_paths.values()):
    raise FileNotFoundError(
        "One or more production artifacts are missing."
    )

print("\nAll production artifacts are available.")

Model: ✓ Found
Preprocessor: ✓ Found
Threshold: ✓ Found

All production artifacts are available.


## Define Feature Engineering

The production prediction system must perform the same feature engineering required by the model.

This is the production version of the feature engineering logic. It should be the single transformation path used by the inference system.

In [5]:
def engineer_features(df):
    """
    Apply the feature engineering required by the churn model.
    """

    data = df.copy()

    # Remove identifier if present
    if "customerID" in data.columns:
        data = data.drop(columns=["customerID"])

    # Convert TotalCharges to numeric
    if "TotalCharges" in data.columns:
        data["TotalCharges"] = pd.to_numeric(
            data["TotalCharges"],
            errors="coerce"
        )

    # Contract duration
    contract_mapping = {
        "Month-to-month": 1,
        "One year": 12,
        "Two year": 24
    }

    if "Contract" in data.columns:
        data["ContractMonths"] = (
            data["Contract"].map(contract_mapping)
        )

    # Total subscribed services
    service_columns = [
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies"
    ]

    available_services = [
        col for col in service_columns
        if col in data.columns
    ]

    if available_services:
        data["TotalServices"] = (
            data[available_services]
            .apply(
                lambda row: sum(
                    value == "Yes"
                    for value in row
                ),
                axis=1
            )
        )

    # Tenure groups
    if "tenure" in data.columns:
        data["TenureGroup"] = pd.cut(
            data["tenure"],
            bins=[-1, 12, 24, 36, 48, 60, 72],
            labels=[
                "0-12 Months",
                "13-24 Months",
                "25-36 Months",
                "37-48 Months",
                "49-60 Months",
                "61-72 Months"
            ]
        )

    return data

# Define Prediction Pipeline

Create the central prediction function.

In [6]:
def predict_churn(
    customer_data,
    model,
    preprocessor,
    threshold
):
    """
    Generate churn predictions for new customer data.

    Returns:
        DataFrame containing churn probability and prediction.
    """

    data = customer_data.copy()

    # Feature engineering
    data_engineered = engineer_features(data)

    # Transform features
    X_processed = preprocessor.transform(data_engineered)

    # Predict probability
    churn_probability = model.predict_proba(
        X_processed
    )[:, 1]

    # Apply production threshold
    churn_prediction = (
        churn_probability >= threshold
    ).astype(int)

    # Create output
    results = data.copy()

    results["Churn_Probability"] = churn_probability

    results["Churn_Prediction"] = churn_prediction

    results["Prediction_Label"] = np.where(
        churn_prediction == 1,
        "Churn",
        "No Churn"
    )

    return results

### Define Risk Segmentation

In [7]:
def assign_risk_level(probability):
    """
    Convert churn probability into business risk level.
    """

    if probability >= 0.70:
        return "High Risk"

    elif probability >= 0.40:
        return "Medium Risk"

    return "Low Risk"


# then 

def add_risk_level(prediction_df):
    """
    Add business risk classification.
    """

    results = prediction_df.copy()

    results["Risk_Level"] = (
        results["Churn_Probability"]
        .apply(assign_risk_level)
    )

    return results

### Define Retention Priority

In [8]:
def add_retention_priority(prediction_df):
    """
    Calculate retention priority using risk level
    and monthly customer value.
    """

    results = prediction_df.copy()

    risk_mapping = {
        "Low Risk": 1,
        "Medium Risk": 2,
        "High Risk": 3
    }

    results["Risk_Score"] = (
        results["Risk_Level"]
        .map(risk_mapping)
    )

    results["Retention_Priority_Score"] = (
        results["Risk_Score"]
        * results["MonthlyCharges"]
    )

    return results

## Define Business Recommendation

This converts prediction into action.

In [9]:
RETENTION_STRATEGY = {
    "High Risk":
        "Immediate retention outreach, personalized offers, "
        "loyalty incentives, and proactive customer support.",

    "Medium Risk":
        "Monitor customer engagement, send personalized "
        "communication, and offer targeted promotions.",

    "Low Risk":
        "Maintain customer relationship through loyalty "
        "programs, regular engagement, and service improvements."
}

# Then

def add_recommended_action(prediction_df):
    """
    Add recommended retention action based on risk level.
    """

    results = prediction_df.copy()

    results["Recommended_Action"] = (
        results["Risk_Level"]
        .map(RETENTION_STRATEGY)
    )

    return results

## The Complete Prediction Function

In [10]:
def run_prediction_pipeline(
    customer_data,
    model,
    preprocessor,
    threshold
):
    """
    Execute the complete production prediction workflow.
    """

    # Generate model predictions
    results = predict_churn(
        customer_data,
        model,
        preprocessor,
        threshold
    )

    # Add business risk
    results = add_risk_level(results)

    # Add retention priority
    results = add_retention_priority(results)

    # Add recommended action
    results = add_recommended_action(results)

    return results

## Create Sample Customer Input

Test the system using new customers.

In [11]:
new_customers = pd.DataFrame({
    "gender": ["Male", "Female", "Male"],
    "SeniorCitizen": [1, 0, 0],
    "Partner": ["No", "Yes", "No"],
    "Dependents": ["No", "Yes", "No"],
    "tenure": [5, 48, 8],
    "PhoneService": ["Yes", "Yes", "Yes"],
    "MultipleLines": ["Yes", "No", "Yes"],
    "InternetService": [
        "Fiber optic",
        "DSL",
        "Fiber optic"
    ],
    "OnlineSecurity": ["No", "Yes", "No"],
    "OnlineBackup": ["No", "Yes", "No"],
    "DeviceProtection": ["No", "Yes", "No"],
    "TechSupport": ["No", "Yes", "No"],
    "StreamingTV": ["Yes", "No", "Yes"],
    "StreamingMovies": ["Yes", "No", "Yes"],
    "Contract": [
        "Month-to-month",
        "One year",
        "Month-to-month"
    ],
    "PaperlessBilling": ["Yes", "No", "Yes"],
    "PaymentMethod": [
        "Electronic check",
        "Credit card (automatic)",
        "Electronic check"
    ],
    "MonthlyCharges": [95.50, 65.25, 101.20],
    "TotalCharges": [477.5, 3132.0, 809.6]
})

new_customers

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Male,1,No,No,5,Yes,Yes,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,95.50,477.5
1,Female,0,Yes,Yes,48,Yes,No,DSL,Yes,Yes,Yes,Yes,No,No,One year,No,Credit card (automatic),65.25,3132.0
2,Male,0,No,No,8,Yes,Yes,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,101.20,809.6


## Run End-to-End Prediction

In [12]:

prediction_results = run_prediction_pipeline(
    customer_data=new_customers,
    model=final_model,
    preprocessor=preprocessor,
    threshold=final_threshold
)

prediction_results

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaymentMethod,MonthlyCharges,TotalCharges,Churn_Probability,Churn_Prediction,Prediction_Label,Risk_Level,Risk_Score,Retention_Priority_Score,Recommended_Action
0,Male,1,No,No,5,Yes,Yes,Fiber optic,No,No,...,Electronic check,95.50,477.5,0.855289,1,Churn,High Risk,3,286.50,"Immediate retention outreach, personalized off..."
1,Female,0,Yes,Yes,48,Yes,No,DSL,Yes,Yes,...,Credit card (automatic),65.25,3132.0,0.033666,0,No Churn,Low Risk,1,65.25,Maintain customer relationship through loyalty...
2,Male,0,No,No,8,Yes,Yes,Fiber optic,No,No,...,Electronic check,101.20,809.6,0.849059,1,Churn,High Risk,3,303.60,"Immediate retention outreach, personalized off..."


# Create Business-Friendly Output that business user actually needs.

In [13]:
business_output = prediction_results[
    [
        "Contract",
        "tenure",
        "MonthlyCharges",
        "InternetService",
        "PaymentMethod",
        "Churn_Probability",
        "Prediction_Label",
        "Risk_Level",
        "Retention_Priority_Score",
        "Recommended_Action"
    ]
].copy()

# Format probability

business_output["Churn_Probability"] = (
    business_output["Churn_Probability"] * 100
).round(2)

# Display

business_output

,Contract,tenure,MonthlyCharges,InternetService,PaymentMethod,Churn_Probability,Prediction_Label,Risk_Level,Retention_Priority_Score,Recommended_Action
0,Month-to-month,5,95.50,Fiber optic,Electronic check,85.529999,Churn,High Risk,286.50,"Immediate retention outreach, personalized off..."
1,One year,48,65.25,DSL,Credit card (automatic),3.370000,No Churn,Low Risk,65.25,Maintain customer relationship through loyalty...
2,Month-to-month,8,101.20,Fiber optic,Electronic check,84.910004,Churn,High Risk,303.60,"Immediate retention outreach, personalized off..."


# Batch Prediction

The model shouldn't only predict one customer. It should be able to process an entire customer dataset.

In [14]:
customer_dataset = pd.read_csv(
    "../data/cleaned/churn_clean.csv"
)

customer_dataset.head()

# Remove target

if "Churn" in customer_dataset.columns:
    customer_dataset = customer_dataset.drop(
        columns=["Churn"]
    )

# Run prediction

batch_predictions = run_prediction_pipeline(
    customer_data=customer_dataset,
    model=final_model,
    preprocessor=preprocessor,
    threshold=final_threshold
)

batch_predictions

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,PaymentMethod,MonthlyCharges,TotalCharges,Churn_Probability,Churn_Prediction,Prediction_Label,Risk_Level,Risk_Score,Retention_Priority_Score,Recommended_Action
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Electronic check,29.85,29.85,0.683173,1,Churn,Medium Risk,2,59.70,"Monitor customer engagement, send personalized..."
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Mailed check,56.95,1889.50,0.035567,0,No Churn,Low Risk,1,56.95,Maintain customer relationship through loyalty...
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Mailed check,53.85,108.15,0.355927,1,Churn,Low Risk,1,53.85,Maintain customer relationship through loyalty...
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Bank transfer (automatic),42.30,1840.75,0.034066,0,No Churn,Low Risk,1,42.30,Maintain customer relationship through loyalty...
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Electronic check,70.70,151.65,0.648511,1,Churn,Medium Risk,2,141.40,"Monitor customer engagement, send personalized..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Mailed check,84.80,1990.50,0.083558,0,No Churn,Low Risk,1,84.80,Maintain customer relationship through loyalty...
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Credit card (automatic),103.20,7362.90,0.124447,0,No Churn,Low Risk,1,103.20,Maintain customer relationship through loyalty...
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,Electronic check,29.60,346.45,0.292591,0,No Churn,Low Risk,1,29.60,Maintain customer relationship through loyalty...
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,Mailed check,74.40,306.60,0.718431,1,Churn,High Risk,3,223.20,"Immediate retention outreach, personalized off..."


# Analyze Batch Predictions

In [15]:
batch_predictions["Risk_Level"].value_counts()

# And

batch_predictions[
    [
        "Churn_Probability",
        "Prediction_Label",
        "Risk_Level"
    ]
].describe(include="all")

# Identify Highest Priority Customers

top_priority_customers = (
    batch_predictions
    .sort_values(
        "Retention_Priority_Score",
        ascending=False
    )
    .head(20)
)

top_priority_customers

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,PaymentMethod,MonthlyCharges,TotalCharges,Churn_Probability,Churn_Prediction,Prediction_Label,Risk_Level,Risk_Score,Retention_Priority_Score,Recommended_Action
1568,3292-PBZEJ,Male,1,No,No,11,Yes,Yes,Fiber optic,No,...,Electronic check,111.40,1183.05,0.719753,1,Churn,High Risk,3,334.20,"Immediate retention outreach, personalized off..."
1176,3992-YWPKO,Female,0,No,No,6,Yes,Yes,Fiber optic,No,...,Credit card (automatic),109.90,669.45,0.736050,1,Churn,High Risk,3,329.70,"Immediate retention outreach, personalized off..."
261,3606-TWKGI,Male,1,No,No,13,Yes,Yes,Fiber optic,No,...,Electronic check,106.90,1364.30,0.779064,1,Churn,High Risk,3,320.70,"Immediate retention outreach, personalized off..."
2294,2027-FECZV,Male,0,No,No,12,Yes,Yes,Fiber optic,No,...,Electronic check,106.70,1253.90,0.769436,1,Churn,High Risk,3,320.10,"Immediate retention outreach, personalized off..."
6894,1400-MMYXY,Male,1,Yes,No,3,Yes,Yes,Fiber optic,No,...,Electronic check,105.90,334.65,0.837279,1,Churn,High Risk,3,317.70,"Immediate retention outreach, personalized off..."
4826,3389-YGYAI,Female,1,No,No,8,Yes,Yes,Fiber optic,No,...,Electronic check,105.50,829.55,0.835232,1,Churn,High Risk,3,316.50,"Immediate retention outreach, personalized off..."
3085,5052-PNLOS,Male,0,No,No,3,Yes,Yes,Fiber optic,No,...,Bank transfer (automatic),105.35,323.25,0.810112,1,Churn,High Risk,3,316.05,"Immediate retention outreach, personalized off..."
2582,7145-FEJWU,Female,0,No,Yes,12,Yes,Yes,Fiber optic,No,...,Electronic check,105.30,1275.65,0.720220,1,Churn,High Risk,3,315.90,"Immediate retention outreach, personalized off..."
3956,4587-VVTOX,Female,0,Yes,No,6,Yes,Yes,Fiber optic,No,...,Electronic check,105.30,545.20,0.832623,1,Churn,High Risk,3,315.90,"Immediate retention outreach, personalized off..."
5933,6496-SLWHQ,Male,1,No,No,3,Yes,Yes,Fiber optic,No,...,Electronic check,105.00,294.45,0.837279,1,Churn,High Risk,3,315.00,"Immediate retention outreach, personalized off..."


### Validate the Prediction Pipeline

Test whether the pipeline behaves correctly.

In [16]:
assert len(prediction_results) == len(new_customers)

assert prediction_results[
    "Churn_Probability"
].between(0, 1).all()

assert prediction_results[
    "Prediction_Label"
].isin(
    ["Churn", "No Churn"]
).all()

assert prediction_results[
    "Risk_Level"
].isin(
    ["Low Risk", "Medium Risk", "High Risk"]
).all()

print("All prediction pipeline checks passed.\n")


# Validate Threshold Logic

expected_predictions = (
    prediction_results["Churn_Probability"]
    >= final_threshold
).astype(int)

assert (
    expected_predictions
    == prediction_results["Churn_Prediction"]
).all()

print("Threshold validation passed.")

All prediction pipeline checks passed.

Threshold validation passed.


## Validate Batch Processing

In [17]:
assert len(batch_predictions) == len(customer_dataset)

assert batch_predictions[
    "Churn_Probability"
].notna().all()

assert batch_predictions[
    "Risk_Level"
].notna().all()

print("Batch prediction validation passed.")

Batch prediction validation passed.


# Save Predictions

The system should produce an actual output artifact.

In [18]:
prediction_output_path = (
    PREDICTION_DIR /
    "customer_churn_predictions.csv"
)

batch_predictions.to_csv(
    prediction_output_path,
    index=False
)

print(
    f"Predictions saved to: "
    f"{prediction_output_path}"
)

Predictions saved to: ..\outputs\predictions\customer_churn_predictions.csv


## Save Top Priority Customers

In [19]:
priority_output_path = (
    PREDICTION_DIR /
    "top_retention_priority_customers.csv"
)

top_priority_customers.to_csv(
    priority_output_path,
    index=False
)

print(
    f"Priority customers saved to: "
    f"{priority_output_path}"
)

Priority customers saved to: ..\outputs\predictions\top_retention_priority_customers.csv


## Production Readiness Check

In [20]:
production_checks = {
    "Model available": MODEL_PATH.exists(),
    "Preprocessor available": PREPROCESSOR_PATH.exists(),
    "Threshold available": THRESHOLD_PATH.exists(),
    "Prediction function available": callable(predict_churn),
    "Risk segmentation available": callable(add_risk_level),
    "Priority scoring available": callable(add_retention_priority),
    "Recommendation engine available": callable(add_recommended_action),
    "Batch prediction successful": len(batch_predictions) > 0,
    "Prediction output saved": prediction_output_path.exists()
}

production_status = pd.DataFrame(
    production_checks.items(),
    columns=["Check", "Status"]
)

production_status

# Then

assert production_status["Status"].all()

print("Production prediction pipeline is ready.")

Production prediction pipeline is ready.


# Final Summary

Notebook 09 establishes the production-oriented inference layer of the customer churn prediction system.

The completed pipeline can:

1. Accept new customer data.
2. Apply the required feature engineering.
3. Transform customer features using the saved preprocessing pipeline.
4. Generate churn probabilities using the production XGBoost model.
5. Apply the optimized classification threshold.
6. Classify customers into churn and non-churn predictions.
7. Segment customers into Low, Medium, and High Risk.
8. Calculate Retention Priority Score.
9. Generate recommended retention actions.
10. Process individual customers or complete customer datasets.
11. Validate prediction outputs.
12. Export prediction results for downstream business use.

The prediction pipeline now provides the machine-learning inference layer required for deployment.

In [21]:
print(type(final_model))
print(final_model)

<class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=1.0, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=True, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=0.5, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.03,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=4, max_leaves=None, min_child_weight=3,
                               missing=nan, monotone_constraints=None,
         